In [79]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import os
import operator

In [80]:
load_dotenv()

True

In [81]:
api_key = os.getenv("GROQ_API")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing from .env")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    groq_api_key=api_key
)

In [82]:
class Evaluator(BaseModel):
    feedback:str = Field(description='Detailed feedback of the essay')
    score: int  = Field(description='Give a score to the quality of the essay from 0 to 10')

In [83]:
structured_model = llm.with_structured_output(Evaluator)


In [84]:
class CSSState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_score: Annotated[list[int], operator.add]
    avg_score: float


In [85]:
def evalThoughts(state:CSSState):
    
    prompt = f"""
Evaluate the clearity of thoughts of the essay.
Give it a score out of 10

Essay:
{state["essay"]}
"""

    result = structured_model.invoke(prompt)
    return {'clarity_feedback':result.feedback, 'individual_score':[result.score]}

In [86]:
def evallang(state:CSSState):
    
    prompt = f"""
Evaluate the language of thoughts of the essay.
Give it a score out of 10

Essay:
{state["essay"]}
"""

    result = structured_model.invoke(prompt)
    return {'language_feedback':result.feedback, 'individual_score':[result.score]}

In [87]:
def evalanalysis(state:CSSState):
    
    prompt = f"""
Evaluate the analysis of the essay.
Give it a score out of 10

Essay:
{state["essay"]}
"""

    result = structured_model.invoke(prompt)
    return {'analysis_feedback':result.feedback, 'individual_score':[result.score]}

In [88]:
def finaleval(state:CSSState):
    
    prompt = f"""
Evaluate the summary on the basis of its analysis, clarity and language
Give it a score out of 10

clarity:
{state["clarity_feedback"]}
analysis:
{state["analysis_feedback"]}
language:
{state["language_feedback"]}
"""

    average = sum(state["individual_score"])/len(state["individual_score"])

    result = llm.invoke(prompt).content
    return {'overall_feedback':result, 'avg_score':average}

In [89]:
graph = StateGraph(CSSState)

graph.add_node('evalThoughts',evalThoughts)
graph.add_node('evalanalysis',evalanalysis)
graph.add_node('evallang',evallang)
graph.add_node('finaleval',finaleval)

In [90]:
graph.add_edge(START,'evallang')
graph.add_edge(START,'evalThoughts')
graph.add_edge(START,'evalanalysis')

graph.add_edge('evallang','finaleval')
graph.add_edge('evalThoughts','finaleval')
graph.add_edge('evalanalysis','finaleval')

graph.add_edge('finaleval',END)

workflow = graph.compile()

workflow.invoke({'essay':'Artificial Intelligence offers Pakistan an important opportunity to overcome development challenges and become more competitive in the global economy. It can transform education, healthcare, agriculture, industry, and public administration. However, achieving these benefits requires sustained investment, skilled professionals, effective policies, and responsible innovation. Pakistan should focus not only on using AI technologies developed by other countries but also on creating its own solutions to local problems. With the right strategy and commitment, Pakistan can develop a stronger AI ecosystem, create employment opportunities for its youth, and contribute meaningfully to the future of artificial intelligence.'
})


{'essay': 'Artificial Intelligence offers Pakistan an important opportunity to overcome development challenges and become more competitive in the global economy. It can transform education, healthcare, agriculture, industry, and public administration. However, achieving these benefits requires sustained investment, skilled professionals, effective policies, and responsible innovation. Pakistan should focus not only on using AI technologies developed by other countries but also on creating its own solutions to local problems. With the right strategy and commitment, Pakistan can develop a stronger AI ecosystem, create employment opportunities for its youth, and contribute meaningfully to the future of artificial intelligence.',
 'language_feedback': 'The essay presents a clear, concise argument about AI’s potential for Pakistan, using straightforward language and logical structure. It effectively balances optimism with realistic conditions, though it could benefit from more specific exam

In [91]:
essay = """
Artificial Intelligence offers Pakistan an important opportunity to overcome development challenges and become more competitive in the global economy. It can transform education, healthcare, agriculture, industry, and public administration. However, achieving these benefits requires sustained investment, skilled professionals, effective policies, and responsible innovation. Pakistan should focus not only on using AI technologies developed by other countries but also on creating its own solutions to local problems. With the right strategy and commitment, Pakistan can develop a stronger AI ecosystem, create employment opportunities for its youth, and contribute meaningfully to the future of artificial intelligence.
"""
prompt = f"""
Evaluate the detail and writing quality of the essay.
Give it a score out of 10 and explain why.

Essay:
{essay}
"""

result = structured_model.invoke(prompt)
print(result)

feedback='The essay is concise and clear, but it lacks depth and specific examples. It outlines broad benefits of AI for Pakistan and emphasizes the need for investment, talent, policy, and local innovation. The writing is coherent, with a logical flow and a balanced tone. However, the piece could be improved by adding concrete data, case studies, or actionable steps, and by expanding on challenges such as infrastructure, data privacy, and workforce development. The language is formal and accessible, but the essay remains somewhat generic and could benefit from more vivid detail and a stronger conclusion. Overall, it demonstrates good structure and clarity but falls short of a comprehensive, richly detailed analysis.\n\nScore: 7' score=7
